### RAG pipeline-data integestion to vector db pipeline

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: eia.pdf
  ✓ Loaded 12 pages

Total documents loaded: 12


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-09-04T14:09:11+05:30', 'author': 'SURYANMURTHY', 'moddate': '2026-09-04T14:09:11+05:30', 'source': '..\\data\\pdf\\eia.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'eia.pdf', 'file_type': 'pdf'}, page_content='Department of Civil Engineering \nIV B. Tech I Semester Question Bank \nSubject: Environmental Impact Assessment Year & Sem: IV & I \nAcademic Year: 2026-27 Branch & Sec: CAM, CAD, CAI & AIML \nName of the Faculty: V. Venkata Jyothi Regulation: VR23 \nUNIT- IV \nAssessment of Impact of development Activities on Vegetation and wildlife, Environmental Impact of \nDeforestation. Environmental Risk Assessment and Risk management in EIA: Risk assessment and \ntreatment of uncertainty -key stages in performing an Environmental Risk Assessment - Advantages of \nEnvironmental Risk Assessment. \n \nQ. No. Question \n1 \nDefine 

In [7]:
### Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split 12 documents into 46 chunks

Example chunk:
Content: Department of Civil Engineering 
IV B. Tech I Semester Question Bank 
Subject: Environmental Impact Assessment Year & Sem: IV & I 
Academic Year: 2026-27 Branch & Sec: CAM, CAD, CAI & AIML 
Name of th...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-09-04T14:09:11+05:30', 'author': 'SURYANMURTHY', 'moddate': '2026-09-04T14:09:11+05:30', 'source': '..\\data\\pdf\\eia.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'eia.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-09-04T14:09:11+05:30', 'author': 'SURYANMURTHY', 'moddate': '2026-09-04T14:09:11+05:30', 'source': '..\\data\\pdf\\eia.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'eia.pdf', 'file_type': 'pdf'}, page_content='Department of Civil Engineering \nIV B. Tech I Semester Question Bank \nSubject: Environmental Impact Assessment Year & Sem: IV & I \nAcademic Year: 2026-27 Branch & Sec: CAM, CAD, CAI & AIML \nName of the Faculty: V. Venkata Jyothi Regulation: VR23 \nUNIT- IV \nAssessment of Impact of development Activities on Vegetation and wildlife, Environmental Impact of \nDeforestation. Environmental Risk Assessment and Risk management in EIA: Risk assessment and \ntreatment of uncertainty -key stages in performing an Environmental Risk Assessment - Advantages of \nEnvironmental Risk Assessment. \n \nQ. No. Question \n1 \nDefine 

###  embedding and vectorstoreDB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14705.83it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_416\3036254068.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [13]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-09-04T14:09:11+05:30', 'author': 'SURYANMURTHY', 'moddate': '2026-09-04T14:09:11+05:30', 'source': '..\\data\\pdf\\eia.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'eia.pdf', 'file_type': 'pdf'}, page_content='Department of Civil Engineering \nIV B. Tech I Semester Question Bank \nSubject: Environmental Impact Assessment Year & Sem: IV & I \nAcademic Year: 2026-27 Branch & Sec: CAM, CAD, CAI & AIML \nName of the Faculty: V. Venkata Jyothi Regulation: VR23 \nUNIT- IV \nAssessment of Impact of development Activities on Vegetation and wildlife, Environmental Impact of \nDeforestation. Environmental Risk Assessment and Risk management in EIA: Risk assessment and \ntreatment of uncertainty -key stages in performing an Environmental Risk Assessment - Advantages of \nEnvironmental Risk Assessment. \n \nQ. No. Question \n1 \nDefine 

In [16]:
### convert the text into embeddings
texts=[doc.page_content for doc in chunks]
### generate embeddings
embeddings=embedding_manager.generate_embeddings(texts)
### store in vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 46 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.02it/s]


Generated embeddings with shape: (46, 384)
Adding 46 documents to vector store...
Successfully added 46 documents to vector store
Total documents in collection: 46


### Retriever Pipeline From VectorStore

In [17]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [18]:
rag_retriever

In [30]:
rag_retriever.retrieve("what is eia?")

Retrieving documents for query: 'what is eia?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_ae7cc947_35',
  'content': '4. Worker and public safety – Helps reduce risks to workers and nearby communities.  \n5. Site selection – Helps identify locations with lower environmental sensitivity and risk.  \n6. Environmental Impact Assessment (EIA)  – Provides risk information for evaluating \nproposed industrial activities.  \n7. Emergency planning  – Supports preparation of emergency response and disaster \nmanagement plans.  \n8. Resource protection  – Assesses potential impacts on soil, surface water, \ngroundwater, vegetation and wildlife.  \n9. Risk mitigation  – Helps select suitable pollution -control technologies and preventive \nmeasures.  \n10. Monitoring and compliance  – Supports continuous environmental monitoring and',
  'metadata': {'page': 8,
   'page_label': '9',
   'creator': 'Microsoft® Word for Microsoft 365',
   'content_length': 705,
   'creationdate': '2026-09-04T14:09:11+05:30',
   'producer': 'Microsoft® Word for Microsoft 365',
   'source': '..

### Integration Vectordb Context pipeline With LLM output

In [40]:
### simple rag pipeline with groq llm
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
llm=ChatGroq(api_key=os.environ.get("GROQ_API_KEY"),model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1024)
def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "no relevant context found to answer the question"
    ## generate the answer using GROQ LLm
    prompt=f"""use the following context to answer the question concisely.
        Context:
        {context}
        Question:{query}

        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [42]:
answer=rag_simple("what is era?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is era?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
no relevant context found to answer the question


### Enhanced RAG Pipeline Features

In [45]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what are risk analysis techniques?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what are risk analysis techniques?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.80it/s]

Generated embeddings with shape: (1, 384)


Retrieved 3 documents (after filtering)
Answer: Based on the context provided, risk analysis techniques include:

*   **Qualitative risk analysis**: Classifies risks as low, medium, or high based on likelihood and consequence.
*   **Quantitative risk analysis**: Uses numerical data to estimate the probability and magnitude of risks.
*   **Risk matrix**: Combines the likelihood of occurrence and severity of consequences to rank risks.
*   **Consequence analysis**: Determines the potential environmental effects of accidents or hazardous events.
*   **Scenario analysis**: Evaluates different possible accident or environmental-impact scenarios.
Sources: [{'source': 'eia.pdf', 'page': 5, 'score': 0.2667163610458374, 'preview': 'event.  \n• Failure Mode and Effects Analysis (FMEA)  – Examines possible equipment or \nprocess failures and their effects.  \n2. Risk Analysis Techniques \n• Qualitative risk analysis  – Classifies risks as low, medium or high  based on \nlikelihood and consequence

In [47]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is risk characterization?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is risk characterization?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Risk characterization  is the process of combining information on hazard, exposure and 
environmental effects to estimate the level and significance of risk. 
1. Risk Characterization Procedure 
1. Combine assessment results – Integrate hazard, exposu

re and effects information.  
2. Estimate risk  – Determine the probability and severity  of adverse environmental 
effects.  
3. Identify affected receptors – Assess impacts on humans, wildlife, vegetation, water, 
soil and ecosystems.  
4. Consider uncertainty – Identify limitations in data, models and assumptions.  
5. Describe risk – Classify the risk as low, moderate or high and explain its significance.  
2. Risk Evaluation Procedure 
Risk evaluation determines whether the identified risk is acceptable or requires further action. 
1. Compare the estimated risk with standards, guidelines or acceptable risk levels.  
2. Identify risks that require control or mitigation.

event.  
• Failure Mode and Effects Analysis (FMEA)  – Examines possible equipment or 
process failures and their effects.  
2. Risk Analysis Techniques 
• Qualitative risk analysis  – Classifies risks as low, medium or high  based on 
likelihood and consequence.  
• Quantitative risk analysis  – Uses numerical dat


Final Answer: Risk characterization is the process of combining information on hazard, exposure, and environmental effects to estimate the level and significance of risk.

Citations:
[1] eia.pdf (page 5)
[2] eia.pdf (page 5)
[3] eia.pdf (page 4)
Summary: Risk characterization integrates data on hazards, exposure, and environmental effects to estimate the overall level and significance of risk. This process serves to synthesize these individual components into a comprehensive assessment of potential risks.
History: {'question': 'what is risk characterization?', 'answer': 'Risk characterization is the process of combining information on hazard, exposure, and environmental effects to estimate the level and significance of risk.', 'sources': [{'source': 'eia.pdf', 'page': 5, 'score': 0.427976131439209, 'preview': 'Risk characterization  is the process of combining information on hazard, exposure and \nenvironmental effects to estimat...'}, {'source': 'eia.pdf', 'page': 5, 'score': 0.25675